# Femur & Hip CT Segmentation: Validation and Femoral-Head Sizing

**What this notebook does.** It runs a published, pretrained nnU-Net bone-segmentation model
(**TotalSegmentator**) on public CT scans that come with expert ground-truth masks. It then
1. measures segmentation accuracy: **Dice, HD95 and ASSD**, in millimetres and spacing-aware;
2. exports the bones as **STL meshes** (3D reconstruction);
3. estimates the **femoral-head diameter and hip centre** by sphere fitting, and measures how
   segmentation error carries into this clinically meaningful number.

**Hypothesis.** The low-resolution `--fast` model (3 mm) will have a Dice similar to the normal model
(1.5 mm), but a clearly larger boundary error (HD95) and femoral-head size error. If so, Dice alone hides
the error that matters for surgical planning.

**There is no training.** This is a *validation study* of an existing model. The long step is
inference: running the model on each CT.

```
CT (NIfTI) ─► TotalSegmentator (normal 1.5 mm + fast 3 mm), ROI: femur_L/R, hip_L/R
                     │
GT masks ────────────┼─► Dice · HD95 · ASSD (mm)
                     ├─► marching cubes ─► STL meshes
                     └─► femoral-head sphere fit ─► diameter & centre (pred vs GT, Bland–Altman)
```

### How to run
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Set the values in **Settings** (the next cell). Keep `N_CASES = 5` for the first run.
3. **Runtime → Run all.** The self-test (section 3) takes about a minute; if it fails, stop and fix it before using real data.
4. The segmentation step (section 6) is the long one. Outputs are saved after every case, so a disconnect loses nothing: just re-run, and finished cases are skipped.

### Time guide (estimates, not measured)
| Step | Rough time |
|---|---|
| Install + write code + self-test | 5–10 min |
| Download (3.2 GB) + unzip | 5–15 min |
| Case selection | 2–5 min |
| TotalSegmentator, per case, both models | measure on the first run (`runtimes.csv`) |
| Evaluation + figures | 2–10 min |

**Data:** TotalSegmentator small subset v2.0.1 (102 CTs with masks), CC BY 4.0. Model and code: Apache-2.0.
Cite: Wasserthal J. et al., *TotalSegmentator: Robust Segmentation of 104 Anatomic Structures in CT Images*, Radiology: AI, 2023.

## 1. Settings
Change these, then run everything below. `N_CASES` is how many CT scans to evaluate. Start with 5,
look at the per-case runtime, then raise it (15–20 is a good final size). Re-running only processes the new cases.

In [ ]:
N_CASES       = 5        # number of CT scans to evaluate (start small)
MODES         = ["full", "fast"]   # full = 1.5 mm model, fast = 3 mm model (the experiment needs both)
EXPORT_STL    = True     # export femur meshes (.stl) for each case
SAVE_TO_DRIVE = True     # copy results to Google Drive so a disconnect never loses them
MIN_VOXELS    = 5000     # a structure must have at least this many GT voxels to count as present

import os, sys, glob, time, shutil, subprocess
IN_COLAB  = "google.colab" in sys.modules
BASE      = "/content" if IN_COLAB else os.path.abspath("colab_local")
REPO      = f"{BASE}/femur-ct-validation"   # project code is written here
DATA_ROOT = f"{BASE}/data"                  # dataset is unzipped here (not in Drive: too big)
DRIVE_OUT = "/content/drive/MyDrive/femur-ct-validation-results"
os.makedirs(REPO, exist_ok=True)
%cd {REPO}
print("Colab:", IN_COLAB, "| working in", REPO)

## 2. Environment
Check the GPU, mount Google Drive (optional), and install packages.

If `pip` prints a warning asking you to **restart the session**, do *Runtime → Restart session*, then
run again from **Settings**. Packages that are already installed are skipped.

In [ ]:
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip())
else:
    print("NO GPU FOUND: Runtime -> Change runtime type -> T4 GPU, then run again from Settings.")

In [ ]:
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print("Results will also be copied to", DRIVE_OUT)

In [ ]:
if IN_COLAB:
    !pip install -q TotalSegmentator nibabel scikit-image trimesh opencv-python-headless tabulate
    !TotalSegmentator --help | head -n 3

## 3. Project code
Each cell below writes one file of the project, so the notebook is self-contained and the same files
can be pushed to GitHub as-is. Read the short explanation before each one; you should be able to explain every function.

In [ ]:
for d in ["femur3d", "scripts", "tests/fake_bin"]:
    os.makedirs(d, exist_ok=True)
open("femur3d/__init__.py", "w").close()

### `femur3d/io.py`: loading NIfTI
Every image is reoriented to the closest canonical (**RAS**) orientation, so CT, ground truth and
predictions share axes: 0 = left–right, 1 = posterior–anterior, 2 = inferior–superior. Voxel spacing (mm)
comes from the header. *Why it matters:* NIfTI/nibabel use RAS, while DICOM/ITK use LPS. Mixing them up silently flips sides.

In [ ]:
%%writefile femur3d/io.py
"""NIfTI loading. Everything is reoriented to the closest canonical (RAS) orientation
so that CT, ground truth and predictions share axes: 0 = left-right,
1 = posterior-anterior, 2 = inferior-superior."""
import nibabel as nib
import numpy as np


def load(path, dtype=None):
    img = nib.as_closest_canonical(nib.load(str(path)))
    arr = np.asanyarray(img.dataobj)
    if dtype is not None:
        arr = arr.astype(dtype)
    return arr, tuple(float(z) for z in img.header.get_zooms()[:3])


def load_mask(path):
    arr, spacing = load(path)
    return arr > 0, spacing

### `femur3d/metrics.py`: accuracy metrics
- **Dice** = 2|A∩B| / (|A|+|B|). It returns NaN if both masks are empty, instead of a misleading 0 or 1.
- **Surface distances.** Surface = mask minus its erosion. A Euclidean distance transform computed with
  `sampling=spacing` gives distances in **mm**, even for anisotropic voxels.
- **HD95** = max of the two directed 95th-percentile surface distances (the MONAI convention).
  **ASSD** = mean symmetric surface distance.
- *Why HD95 as well as Dice:* Dice is dominated by the bulk volume of a big bone, while HD95 reflects
  boundary errors, which are what move a measurement.

In [ ]:
%%writefile femur3d/metrics.py
"""Overlap and surface-distance metrics for binary 3D masks.

All distances are in millimetres: voxel spacing is passed in explicitly,
so anisotropic CT voxels are handled correctly.
"""
import numpy as np
from scipy import ndimage


def dice(gt: np.ndarray, pred: np.ndarray) -> float:
    """Dice similarity coefficient. Returns nan if both masks are empty."""
    gt, pred = gt.astype(bool), pred.astype(bool)
    denom = gt.sum() + pred.sum()
    if denom == 0:
        return float("nan")
    return 2.0 * np.logical_and(gt, pred).sum() / denom


def _surface(mask: np.ndarray) -> np.ndarray:
    """Boundary voxels: voxels in the mask that touch the background (6-connectivity)."""
    eroded = ndimage.binary_erosion(mask, structure=ndimage.generate_binary_structure(3, 1))
    return np.logical_and(mask, ~eroded)


def _directed_distances(src_surf, dst_surf, spacing):
    """Distance (mm) from every surface voxel of src to the nearest surface voxel of dst."""
    # EDT of the complement of dst's surface gives, at each voxel, distance to dst's surface.
    dt = ndimage.distance_transform_edt(~dst_surf, sampling=spacing)
    return dt[src_surf]


def surface_metrics(gt: np.ndarray, pred: np.ndarray, spacing) -> dict:
    """HD95 and ASSD in mm.

    HD95 follows the MONAI convention: max of the two directed 95th percentiles.
    ASSD = mean of all symmetric surface distances.
    Returns nan values if either mask is empty.
    """
    gt, pred = gt.astype(bool), pred.astype(bool)
    if gt.sum() == 0 or pred.sum() == 0:
        return {"hd95": float("nan"), "assd": float("nan")}
    # Crop to the joint bounding box (plus margin) for speed.
    sl = _joint_bbox(gt | pred, margin=3)
    gt, pred = gt[sl], pred[sl]
    s_gt, s_pr = _surface(gt), _surface(pred)
    d_gp = _directed_distances(s_gt, s_pr, spacing)
    d_pg = _directed_distances(s_pr, s_gt, spacing)
    hd95 = max(np.percentile(d_gp, 95), np.percentile(d_pg, 95))
    assd = np.concatenate([d_gp, d_pg]).mean()
    return {"hd95": float(hd95), "assd": float(assd)}


def _joint_bbox(mask, margin=0):
    idx = np.argwhere(mask)
    lo = np.maximum(idx.min(0) - margin, 0)
    hi = np.minimum(idx.max(0) + margin + 1, mask.shape)
    return tuple(slice(a, b) for a, b in zip(lo, hi))

### `femur3d/geometry.py`: meshes and femoral-head fitting
- `mask_to_mesh`: marching cubes at the 0.5 iso-level with real spacing (vertices in mm). It fixes the face
  winding if the volume comes out negative, then applies Taubin smoothing (smooths without shrinking) and exports STL.
- `fit_sphere`: linear least squares. ‖p−c‖² = r² becomes x²+y²+z² = 2c·p + d, which is linear in (c, d); then r = √(d + ‖c‖²).
- `femoral_head`:
  1. Take the femur surface points within **6 mm of the hip bone**. That is the head facing the acetabulum, since the joint space is only a few mm.
  2. Fit a sphere to them.
  3. Refine three times using all femur surface points within **±2 mm** of the sphere. This pulls in the rest of the head but not the narrower neck.

  Surface points are sub-voxel marching-cubes vertices. Voxel centres biased the radius low by about half a voxel in testing.

In [ ]:
%%writefile femur3d/geometry.py
"""3D geometry: mask -> STL mesh, and femoral-head sphere fitting.

Femoral-head size is a planning input in hip arthroplasty (head/cup sizing)
and the hip centre is one end of the femoral mechanical axis.
Here it is estimated from the segmentation alone, so errors in the
segmentation propagate into a clinically meaningful number.
"""
import numpy as np
from scipy import ndimage
from skimage import measure
import trimesh


def mask_to_mesh(mask: np.ndarray, spacing, smooth_iters: int = 10) -> trimesh.Trimesh:
    """Marching cubes on a binary mask; vertices in mm (grid coordinates * spacing)."""
    padded = np.pad(mask.astype(np.uint8), 1)  # pad so the surface closes at volume edges
    verts, faces, _, _ = measure.marching_cubes(padded, level=0.5, spacing=tuple(spacing))
    verts -= np.asarray(spacing)  # undo the 1-voxel pad offset
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
    if mesh.volume < 0:  # marching-cubes winding depends on axis order; make normals point outward
        mesh.invert()
    if smooth_iters > 0:
        trimesh.smoothing.filter_taubin(mesh, iterations=smooth_iters)  # volume-preserving smoothing
    return mesh


def surface_points(mask: np.ndarray, spacing) -> np.ndarray:
    """Sub-voxel surface points (mm) from marching cubes at the 0.5 iso-level.
    Voxel centres of boundary voxels sit ~half a voxel inside the true surface and
    bias the fitted radius low; marching-cubes vertices do not."""
    sl = _bbox(mask, margin=2)
    offset = np.array([s.start for s in sl]) * np.asarray(spacing)
    verts, _, _, _ = measure.marching_cubes(np.pad(mask[sl].astype(np.uint8), 1), level=0.5,
                                            spacing=tuple(spacing))
    return verts - np.asarray(spacing) + offset


def _bbox(mask, margin=0):
    idx = np.argwhere(mask)
    lo = np.maximum(idx.min(0) - margin, 0)
    hi = np.minimum(idx.max(0) + margin + 1, mask.shape)
    return tuple(slice(a, b) for a, b in zip(lo, hi))


def fit_sphere(points: np.ndarray):
    """Algebraic least-squares sphere fit. points: (N, 3) in mm. Returns (centre, radius, rmse)."""
    A = np.c_[2 * points, np.ones(len(points))]
    b = (points ** 2).sum(1)
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    centre = sol[:3]
    radius = np.sqrt(sol[3] + centre @ centre)
    resid = np.linalg.norm(points - centre, axis=1) - radius
    return centre, float(radius), float(np.sqrt(np.mean(resid ** 2)))


def femoral_head(femur: np.ndarray, hip: np.ndarray, spacing,
                 contact_mm: float = 6.0, band_mm: float = 2.0, n_refine: int = 3,
                 min_points: int = 50):
    """Estimate femoral-head centre and diameter from femur + hip-bone masks.

    1. Femur surface voxels within `contact_mm` of the hip bone = the part of the
       head that faces the acetabulum (joint space is only a few mm).
    2. Fit a sphere to those points.
    3. Refine: refit using all femur surface points within `band_mm` of the sphere,
       which pulls in the rest of the head but not the narrower neck.
    Returns dict (centre_mm, diameter_mm, rmse_mm, n_points) or None on failure.
    """
    femur, hip = femur.astype(bool), hip.astype(bool)
    if femur.sum() == 0 or hip.sum() == 0:
        return None
    spacing = np.asarray(spacing, float)
    # Restrict work to the region around the hip bone (the head lies next to it).
    sl = _bbox(hip, margin=int(np.ceil(60 / spacing.min())))
    offset = np.array([s.start for s in sl]) * spacing
    femur_c, hip_c = femur[sl], hip[sl]
    if femur_c.sum() == 0:
        return None
    dist_to_hip = ndimage.distance_transform_edt(~hip_c, sampling=spacing)
    surf_pts = surface_points(femur_c, spacing)                      # mm, local to the crop
    vox = np.clip(np.rint(surf_pts / spacing).astype(int), 0, np.array(femur_c.shape) - 1)
    d_hip = dist_to_hip[vox[:, 0], vox[:, 1], vox[:, 2]]
    contact = surf_pts[d_hip < contact_mm]
    if len(contact) < min_points:
        return None
    centre, r, rmse = fit_sphere(contact)
    pts = contact
    for _ in range(n_refine):
        d = np.linalg.norm(surf_pts - centre, axis=1)
        sel = surf_pts[np.abs(d - r) < band_mm]
        if len(sel) < min_points:
            break
        centre, r, rmse = fit_sphere(sel)
        pts = sel
    return {"centre_mm": centre + offset, "diameter_mm": 2 * r, "rmse_mm": rmse, "n_points": int(len(pts))}

### `scripts/select_cases.py`: choosing CT scans
It keeps scans where all four structures (both femurs, both hip bones) are present in the ground truth, and
takes the **official test split first**. TotalSegmentator was trained on the train split, so evaluating on
it would be optimistic (train/test leakage). Any non-test case is labelled in the output.

In [ ]:
%%writefile scripts/select_cases.py
"""Pick CT cases whose ground truth contains both femoral heads and both hip bones.

Prefers the official test split (meta.csv 'split' column) because TotalSegmentator
was trained on the train split; evaluating on training cases would be optimistic.

Usage: python scripts/select_cases.py DATA_DIR --n 15 --out cases.csv
"""
import argparse
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd

STRUCTS = ["femur_left", "femur_right", "hip_left", "hip_right"]


def has_structures(case_dir: Path, min_voxels: int) -> bool:
    for s in STRUCTS:
        p = case_dir / "segmentations" / f"{s}.nii.gz"
        if not p.exists():
            return False
        if np.count_nonzero(np.asanyarray(nib.load(str(p)).dataobj)) < min_voxels:
            return False
    return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data_dir", type=Path)
    ap.add_argument("--n", type=int, default=15)
    ap.add_argument("--min_voxels", type=int, default=5000)
    ap.add_argument("--out", type=Path, default=Path("cases.csv"))
    a = ap.parse_args()

    meta_path = a.data_dir / "meta.csv"
    meta = pd.read_csv(meta_path, sep=None, engine="python") if meta_path.exists() else None
    cases = sorted(p.name for p in a.data_dir.iterdir() if (p / "ct.nii.gz").exists())
    df = pd.DataFrame({"image_id": cases})
    if meta is not None and "split" in meta.columns:
        df = df.merge(meta[["image_id", "split"]], on="image_id", how="left")
    else:
        df["split"] = "unknown"
    print(df["split"].value_counts(dropna=False).to_string())

    # Test cases first, then validation, then train (flagged in the output).
    order = {"test": 0, "val": 1, "validation": 1, "train": 2}
    df["rank"] = df["split"].map(order).fillna(3)
    df = df.sort_values(["rank", "image_id"])

    keep = []
    for _, row in df.iterrows():
        if has_structures(a.data_dir / row.image_id, a.min_voxels):
            keep.append(row)
            print("keep", row.image_id, row.split)
        if len(keep) >= a.n:
            break
    out = pd.DataFrame(keep)[["image_id", "split"]]
    out.to_csv(a.out, index=False)
    print(f"{len(out)} cases written to {a.out}; splits: {out.split.value_counts().to_dict()}")


if __name__ == "__main__":
    main()

### `scripts/run_totalseg.py`: running the model
It calls the `TotalSegmentator` command line once per case per model, with `--roi_subset femur_left femur_right
hip_left hip_right` (and `--fast` for the 3 mm model). It logs the runtime of every call to `preds/runtimes.csv`
and skips cases that are already done.

In [ ]:
%%writefile scripts/run_totalseg.py
"""Run TotalSegmentator (normal 1.5 mm and --fast 3 mm models) on the selected cases.

Usage: python scripts/run_totalseg.py DATA_DIR cases.csv --out preds
Writes preds/<mode>/<case>/<structure>.nii.gz and preds/runtimes.csv
"""
import argparse
import subprocess
import time
from pathlib import Path
import pandas as pd

ROI = ["femur_left", "femur_right", "hip_left", "hip_right"]
MODES = {"full": [], "fast": ["--fast"]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data_dir", type=Path)
    ap.add_argument("cases", type=Path)
    ap.add_argument("--out", type=Path, default=Path("preds"))
    ap.add_argument("--modes", nargs="+", default=list(MODES))
    a = ap.parse_args()

    rows = []
    for case in pd.read_csv(a.cases).image_id:
        for mode in a.modes:
            out_dir = a.out / mode / case
            if (out_dir / "femur_left.nii.gz").exists():
                print("skip (done)", mode, case)
                continue
            out_dir.mkdir(parents=True, exist_ok=True)
            cmd = ["TotalSegmentator", "-i", str(a.data_dir / case / "ct.nii.gz"),
                   "-o", str(out_dir), "--roi_subset", *ROI, *MODES[mode]]
            t0 = time.time()
            res = subprocess.run(cmd, capture_output=True, text=True)
            dt = time.time() - t0
            ok = res.returncode == 0
            print(f"{mode:4s} {case}: {'ok' if ok else 'FAILED'} in {dt:.1f}s")
            if not ok:
                print(res.stderr[-2000:])
            rows.append({"image_id": case, "mode": mode, "seconds": round(dt, 1), "ok": ok})
            # Append after every run so a Colab disconnect loses nothing.
            pd.DataFrame(rows).to_csv(a.out / "runtimes.csv", index=False)


if __name__ == "__main__":
    main()

### `scripts/evaluate.py`: comparing with ground truth
For every case, model and bone it computes Dice / HD95 / ASSD. For every side it computes GT vs predicted
femoral-head diameter and centre error. It writes `results/segmentation_metrics.csv`, `results/femoral_head.csv`
and `results/summary.md` (tables ready for the README), plus optional STL meshes.

In [ ]:
%%writefile scripts/evaluate.py
"""Compare predictions with ground truth: Dice, HD95, ASSD per bone, and
femoral-head diameter / centre error per side. Also exports STL meshes.

Usage: python scripts/evaluate.py DATA_DIR cases.csv --preds preds --out results
"""
import argparse
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from femur3d.io import load_mask  # noqa: E402
from femur3d.metrics import dice, surface_metrics  # noqa: E402
from femur3d.geometry import femoral_head, mask_to_mesh  # noqa: E402

BONES = ["femur_left", "femur_right", "hip_left", "hip_right"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data_dir", type=Path)
    ap.add_argument("cases", type=Path)
    ap.add_argument("--preds", type=Path, default=Path("preds"))
    ap.add_argument("--out", type=Path, default=Path("results"))
    ap.add_argument("--modes", nargs="+", default=["full", "fast"])
    ap.add_argument("--stl", action="store_true", help="export STL meshes for each case")
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)

    seg_rows, head_rows = [], []
    cases = pd.read_csv(a.cases)
    for case, split in zip(cases.image_id, cases.split):
        gt = {}
        for b in BONES:
            gt[b], spacing = load_mask(a.data_dir / case / "segmentations" / f"{b}.nii.gz")
        gt_head = {s: femoral_head(gt[f"femur_{s}"], gt[f"hip_{s}"], spacing) for s in ("left", "right")}
        if a.stl:
            (a.out / "stl" / case).mkdir(parents=True, exist_ok=True)
            mask_to_mesh(gt["femur_left"], spacing).export(a.out / "stl" / case / "femur_left_gt.stl")

        for mode in a.modes:
            pdir = a.preds / mode / case
            if not (pdir / "femur_left.nii.gz").exists():
                print("missing prediction", mode, case)
                continue
            pr = {b: load_mask(pdir / f"{b}.nii.gz")[0] for b in BONES}
            for b in BONES:
                row = {"image_id": case, "split": split, "mode": mode, "bone": b,
                       "dice": dice(gt[b], pr[b]), **surface_metrics(gt[b], pr[b], spacing)}
                seg_rows.append(row)
            for s in ("left", "right"):
                ph = femoral_head(pr[f"femur_{s}"], pr[f"hip_{s}"], spacing)
                g = gt_head[s]
                row = {"image_id": case, "split": split, "mode": mode, "side": s,
                       "gt_diam_mm": g["diameter_mm"] if g else np.nan,
                       "pred_diam_mm": ph["diameter_mm"] if ph else np.nan,
                       "gt_fit_rmse_mm": g["rmse_mm"] if g else np.nan,
                       "centre_err_mm": (np.linalg.norm(g["centre_mm"] - ph["centre_mm"])
                                         if (g and ph) else np.nan)}
                row["diam_err_mm"] = row["pred_diam_mm"] - row["gt_diam_mm"]
                head_rows.append(row)
            if a.stl:
                mask_to_mesh(pr["femur_left"], spacing).export(a.out / "stl" / case / f"femur_left_{mode}.stl")
            print("done", case, mode)

        # Save incrementally.
        pd.DataFrame(seg_rows).to_csv(a.out / "segmentation_metrics.csv", index=False)
        pd.DataFrame(head_rows).to_csv(a.out / "femoral_head.csv", index=False)

    seg = pd.DataFrame(seg_rows)
    head = pd.DataFrame(head_rows)
    summary = seg.groupby(["mode", "bone"])[["dice", "hd95", "assd"]].agg(["mean", "std", "median"]).round(3)
    head_summary = head.groupby("mode").agg(
        n=("diam_err_mm", "count"),
        mean_diam_err=("diam_err_mm", "mean"),
        sd_diam_err=("diam_err_mm", "std"),
        mae_diam=("diam_err_mm", lambda x: x.abs().mean()),
        mean_centre_err=("centre_err_mm", "mean"),
    ).round(2)
    print(summary.to_string())
    print(head_summary.to_string())
    with open(a.out / "summary.md", "w") as f:
        f.write("## Segmentation (Dice, HD95 mm, ASSD mm)\n\n" + summary.to_markdown() + "\n\n")
        f.write("## Femoral-head diameter error (pred - GT, mm) and centre error (mm)\n\n"
                + head_summary.to_markdown() + "\n")


if __name__ == "__main__":
    main()

### `scripts/make_figures.py`: figures
- **OpenCV QC overlay:** GT (green) and prediction (red) contours on a coronal CT slice through the fitted head centre, using a bone window and square pixels.
- **Bland–Altman plot** of femoral-head diameter (bias and 95% limits of agreement).
- **Dice boxplot** per bone and model.

In [ ]:
%%writefile scripts/make_figures.py
"""Figures: (1) OpenCV QC overlay of GT vs prediction contours on a coronal CT slice
through the femoral head, (2) Bland-Altman plot of femoral-head diameter, (3) Dice boxplot.

Usage: python scripts/make_figures.py DATA_DIR --case s0011 --preds preds --results results
"""
import argparse
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from femur3d.io import load, load_mask  # noqa: E402
from femur3d.geometry import femoral_head  # noqa: E402


def window(ct_slice, lo=-400, hi=1500):
    """Bone window -> uint8."""
    return (np.clip((ct_slice - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)


def to_display(a):
    """Coronal slice arr[x, z] -> image with superior at the top."""
    return np.ascontiguousarray(np.rot90(a))


def draw(img_bgr, mask2d, colour):
    cnts, _ = cv2.findContours(mask2d.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(img_bgr, cnts, -1, colour, 2)


def qc_overlay(data_dir, case, preds, mode, out_png, side="left"):
    ct, spacing = load(data_dir / case / "ct.nii.gz", np.float32)
    gt, _ = load_mask(data_dir / case / "segmentations" / f"femur_{side}.nii.gz")
    hip, _ = load_mask(data_dir / case / "segmentations" / f"hip_{side}.nii.gz")
    pr, _ = load_mask(preds / mode / case / f"femur_{side}.nii.gz")
    head = femoral_head(gt, hip, spacing)
    y = int(round(head["centre_mm"][1] / spacing[1])) if head else gt.shape[1] // 2
    gray = to_display(window(ct[:, y, :]))
    m_gt, m_pr = to_display(gt[:, y, :]).astype(np.uint8), to_display(pr[:, y, :]).astype(np.uint8)
    # Resample to square pixels (3 px per mm) BEFORE drawing, so anatomy is not stretched
    # and contours stay crisp.
    h, w = gray.shape
    size = (int(w * spacing[0] * 3), int(h * spacing[2] * 3))
    gray = cv2.resize(gray, size, interpolation=cv2.INTER_LINEAR)
    m_gt = cv2.resize(m_gt, size, interpolation=cv2.INTER_NEAREST)
    m_pr = cv2.resize(m_pr, size, interpolation=cv2.INTER_NEAREST)
    img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    draw(img, m_gt, (0, 255, 0))   # green = ground truth
    draw(img, m_pr, (0, 0, 255))   # red = prediction
    cv2.putText(img, f"{case} {mode} | green=GT red=pred | slice y={y}", (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.imwrite(str(out_png), img)


def bland_altman(head_csv, out_png):
    df = pd.read_csv(head_csv).dropna(subset=["gt_diam_mm", "pred_diam_mm"])
    fig, axes = plt.subplots(1, df["mode"].nunique(), figsize=(10, 4), squeeze=False)
    for ax, (mode, d) in zip(axes[0], df.groupby("mode")):
        mean = (d.gt_diam_mm + d.pred_diam_mm) / 2
        diff = d.pred_diam_mm - d.gt_diam_mm
        bias, sd = diff.mean(), diff.std()
        ax.scatter(mean, diff, s=18)
        for v, ls in [(bias, "-"), (bias + 1.96 * sd, "--"), (bias - 1.96 * sd, "--")]:
            ax.axhline(v, ls=ls, c="k", lw=1)
        ax.ticklabel_format(useOffset=False)
        ax.set(title=f"{mode}: bias {bias:.2f} mm, LoA ±{1.96*sd:.2f}",
               xlabel="mean head diameter (mm)", ylabel="pred - GT (mm)")
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)


def dice_box(seg_csv, out_png):
    df = pd.read_csv(seg_csv)
    fig, ax = plt.subplots(figsize=(8, 4))
    labels, data = [], []
    for (bone, mode), d in df.groupby(["bone", "mode"]):
        labels.append(f"{bone}\n{mode}")
        data.append(d.dice.dropna())
    ax.boxplot(data)
    ax.set_xticks(range(1, len(labels) + 1), labels, fontsize=7)
    ax.set_ylabel("Dice")
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("data_dir", type=Path)
    ap.add_argument("--case", required=True)
    ap.add_argument("--preds", type=Path, default=Path("preds"))
    ap.add_argument("--results", type=Path, default=Path("results"))
    a = ap.parse_args()
    fig_dir = a.results / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    for mode in ("full", "fast"):
        if (a.preds / mode / a.case).exists():
            qc_overlay(a.data_dir, a.case, a.preds, mode, fig_dir / f"qc_{a.case}_{mode}.png")
    bland_altman(a.results / "femoral_head.csv", fig_dir / "bland_altman_head_diameter.png")
    dice_box(a.results / "segmentation_metrics.csv", fig_dir / "dice_boxplot.png")
    print("figures in", fig_dir)


if __name__ == "__main__":
    main()

### `tests/`: checking the code without real data
- `make_synthetic.py` builds a phantom (spherical head of **known 46.0 mm diameter** + neck + shaft + acetabular shell).
- `fake_bin/TotalSegmentator` imitates the model: it erodes the GT for "normal" and dilates it for "fast".

This verifies the measurement code before trusting it on real scans.

In [ ]:
%%writefile tests/make_synthetic.py
"""Build a tiny synthetic 'TotalSegmentator-like' dataset to test the pipeline offline.
Femoral head = sphere of known radius, so the sphere fit has a known answer."""
import sys
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd

root = Path(sys.argv[1])
SPACING = np.array([0.8, 0.8, 1.5])
SHAPE = (200, 90, 110)
R_HEAD = 23.0  # mm -> true diameter 46 mm


def grid():
    idx = np.indices(SHAPE).reshape(3, -1).T * SPACING
    return idx.reshape(*SHAPE, 3)


def side(P, cx, sign):
    c = np.array([cx, 36.0, 100.0])
    d = np.linalg.norm(P - c, axis=-1)
    head = d <= R_HEAD
    # neck: cylinder from head centre going lateral (sign) and down
    axis = np.array([sign * 0.7, 0.0, -0.7]); axis /= np.linalg.norm(axis)
    v = P - c
    t = v @ axis
    radial = np.linalg.norm(v - t[..., None] * axis, axis=-1)
    neck = (t > 0) & (t < 45) & (radial < 13)
    # shaft: vertical cylinder below the neck end
    s_c = c + 45 * axis
    shaft = (np.linalg.norm((P - s_c)[..., :2], axis=-1) < 12) & (P[..., 2] < s_c[2])
    femur = head | neck | shaft
    # acetabulum: shell around the head covering the upper/medial part, plus a block above
    shell = (d > R_HEAD + 2.5) & (d < R_HEAD + 10) & (P[..., 2] > c[2] - 4)
    block = (np.abs(P[..., 0] - cx) < 30) & (P[..., 2] > c[2] + R_HEAD + 8) & (P[..., 2] < c[2] + R_HEAD + 30)
    hip = (shell | block) & ~femur
    return femur, hip


def main():
    P = grid()
    affine = np.diag([*SPACING, 1.0])
    rng = np.random.default_rng(0)
    rows = []
    for case, split in [("s0001", "test"), ("s0002", "train"), ("s0003", "test")]:
        d = root / case / "segmentations"
        d.mkdir(parents=True, exist_ok=True)
        fl, hl = side(P, 50.0, +1)
        fr, hr = side(P, 110.0, -1)
        ct = rng.normal(0, 30, SHAPE).astype(np.float32)
        ct[fl | fr] = 900
        ct[hl | hr] = 700
        nib.save(nib.Nifti1Image(ct, affine), str(root / case / "ct.nii.gz"))
        for name, m in [("femur_left", fl), ("femur_right", fr), ("hip_left", hl), ("hip_right", hr)]:
            nib.save(nib.Nifti1Image(m.astype(np.uint8), affine), str(d / f"{name}.nii.gz"))
        rows.append({"image_id": case, "split": split})
    pd.DataFrame(rows).to_csv(root / "meta.csv", sep=";", index=False)
    print("synthetic dataset at", root)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tests/fake_bin/TotalSegmentator
#!/usr/bin/env python3
"""Fake TotalSegmentator for offline testing: copies GT masks, eroded once
(normal mode) or dilated twice (--fast) so metrics are non-trivial."""
import sys
from pathlib import Path
import numpy as np
import nibabel as nib
from scipy import ndimage

args = sys.argv[1:]
inp = Path(args[args.index("-i") + 1])
out = Path(args[args.index("-o") + 1])
fast = "--fast" in args
roi = args[args.index("--roi_subset") + 1:]
roi = [r for r in roi if not r.startswith("--")]
out.mkdir(parents=True, exist_ok=True)
for name in roi:
    img = nib.load(str(inp.parent / "segmentations" / f"{name}.nii.gz"))
    m = np.asanyarray(img.dataobj) > 0
    m = ndimage.binary_dilation(m, iterations=2) if fast else ndimage.binary_erosion(m)
    nib.save(nib.Nifti1Image(m.astype(np.uint8), img.affine), str(out / f"{name}.nii.gz"))

In [ ]:
os.chmod("tests/fake_bin/TotalSegmentator", 0o755)
open("requirements.txt", "w").write("TotalSegmentator>=2.0\nnibabel\nnumpy\nscipy\npandas\nscikit-image\ntrimesh\n"
                                    "opencv-python-headless\nmatplotlib\ntabulate\n")
print(sorted(glob.glob("**/*.py", recursive=True)))

## 4. Self-test on a synthetic phantom (about 1 minute)
This runs the whole pipeline on fake data with a known answer. **Expected:** the GT femoral-head diameter is about **46.0 mm**
(within ±0.5 mm); "full" (eroded) gives a smaller diameter and "fast" (dilated) a larger one; Dice is below 1 for both.
If this fails, do not continue.

In [ ]:
import pandas as pd
ST = f"{BASE}/selftest"
shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST)
env = dict(os.environ, PATH=f"{REPO}/tests/fake_bin:" + os.environ["PATH"])
def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f"failed: {cmd}")
    return r.stdout
sh(f"python tests/make_synthetic.py {ST}/data")
sh(f"python scripts/select_cases.py {ST}/data --n 2 --min_voxels 100 --out {ST}/cases.csv")
sh(f"python scripts/run_totalseg.py {ST}/data {ST}/cases.csv --out {ST}/preds", env=env)
sh(f"python scripts/evaluate.py {ST}/data {ST}/cases.csv --preds {ST}/preds --out {ST}/results")
h = pd.read_csv(f"{ST}/results/femoral_head.csv")
display(h[["image_id", "mode", "side", "gt_diam_mm", "pred_diam_mm", "diam_err_mm"]].round(2))
gt = h.gt_diam_mm.mean()
ok = abs(gt - 46.0) < 0.5 and (h[h["mode"] == "full"].diam_err_mm < 0).all() and (h[h["mode"] == "fast"].diam_err_mm > 0).all()
print(f"GT diameter {gt:.2f} mm (truth 46.0) ->", "SELF-TEST PASSED" if ok else "SELF-TEST FAILED")
assert ok

## 5. Download the dataset (3.2 GB)
This tries the Dropbox mirror first (usually faster), then Zenodo. The data stays in `/content` (fast local disk), not in
Drive. If the session resets, this cell downloads again (it skips the download if the zip is still there).

In [ ]:
ZIP = f"{BASE}/ds.zip"
URLS = [
    "https://www.dropbox.com/scl/fi/pee5yxebfxrhz007cbuy5/Totalsegmentator_dataset_small_v201.zip?rlkey=osvfk02jc4lw5gr6uhrldtb9e&dl=1",
    "https://zenodo.org/records/10047263/files/Totalsegmentator_dataset_small_v201.zip?download=1",
]
if not glob.glob(f"{DATA_ROOT}/**/meta.csv", recursive=True):
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 3e9:
        for url in URLS:
            print("downloading from", url.split("/")[2], "...")
            if subprocess.run(["wget", "-q", "--show-progress", "-O", ZIP, url]).returncode == 0 and os.path.getsize(ZIP) > 3e9:
                break
    print("unzipping ...")
    subprocess.run(["unzip", "-q", "-o", ZIP, "-d", DATA_ROOT], check=True)
meta = glob.glob(f"{DATA_ROOT}/**/meta.csv", recursive=True)
DATA = os.path.dirname(meta[0]) if meta else DATA_ROOT
print("DATA =", DATA, "| subjects:", len(glob.glob(f"{DATA}/s*/ct.nii.gz")))

**Check the layout.** You should see folders like `s0000/` containing `ct.nii.gz` and `segmentations/femur_left.nii.gz`, and a
`meta.csv` with a `split` column. If the names differ, fix them before continuing; the scripts assume this layout.

In [ ]:
m = pd.read_csv(f"{DATA}/meta.csv", sep=None, engine="python")
print("meta.csv columns:", list(m.columns))
if "split" in m.columns: print(m["split"].value_counts().to_string())
first = sorted(glob.glob(f"{DATA}/s*"))[0]
print(first, "->", sorted(os.listdir(first)))
print("femur/hip masks present:", [os.path.basename(p) for p in sorted(glob.glob(f"{first}/segmentations/*")) if "femur" in p or "hip_" in p])

## 6. Select cases and run TotalSegmentator (the long step)
On the very first call, TotalSegmentator downloads its model weights. Look at `runtimes.csv` after the first case
or two. If each run is short, raise `N_CASES` in Settings and re-run sections 6–9.

**If `--fast` together with `--roi_subset` errors:** open `scripts/run_totalseg.py`, remove `"--roi_subset", *ROI,`
for the fast mode only, and re-run.

In [ ]:
!python scripts/select_cases.py {DATA} --n {N_CASES} --min_voxels {MIN_VOXELS} --out cases.csv
cases = pd.read_csv("cases.csv"); display(cases)
if (cases.split != "test").any():
    print("NOTE: some cases are not from the test split; label them as optimistic in the README.")

In [ ]:
t0 = time.time()
!python scripts/run_totalseg.py {DATA} cases.csv --out preds --modes {" ".join(MODES)}
print(f"total {(time.time() - t0) / 60:.1f} min")
display(pd.read_csv("preds/runtimes.csv").groupby("mode").seconds.describe().round(1))

## 7. Evaluate against ground truth

In [ ]:
stl = "--stl" if EXPORT_STL else ""
!python scripts/evaluate.py {DATA} cases.csv --preds preds --out results --modes {" ".join(MODES)} {stl} | tail -n 30
seg  = pd.read_csv("results/segmentation_metrics.csv")
head = pd.read_csv("results/femoral_head.csv")
display(seg.groupby(["mode", "bone"])[["dice", "hd95", "assd"]].agg(["mean", "std"]).round(3))
display(head.groupby("mode")[["diam_err_mm", "centre_err_mm"]].agg(["count", "mean", "std"]).round(2))

## 8. Sanity checks (do not skip)
These are automatic warnings. Investigate every one before you write anything up.
- GT femoral-head diameters outside about 35–60 mm (published adult means are around 45–48 mm) → wrong side, truncated head, orientation issue, or a paediatric case.
- A large fit RMSE → the sphere fit grabbed the neck or a partial head.
- A failed fit (NaN) → a structure is missing or the contact region was too small.
- Dice < 0.8 on a femur → wrong side, a cropped scan, or an implant. Look at the image.

In [ ]:
warn = 0
for _, r in head[head["mode"] == MODES[0]].iterrows():
    if pd.isna(r.gt_diam_mm):
        print(f"[fit failed] {r.image_id} {r.side}"); warn += 1
    elif not 35 <= r.gt_diam_mm <= 60:
        print(f"[implausible GT diameter {r.gt_diam_mm:.1f} mm] {r.image_id} {r.side}"); warn += 1
    elif r.gt_fit_rmse_mm > 1.5:
        print(f"[poor sphere fit, rmse {r.gt_fit_rmse_mm:.2f} mm] {r.image_id} {r.side}"); warn += 1
for _, r in seg[seg.bone.str.startswith("femur")].iterrows():
    if r.dice < 0.8:
        print(f"[low Dice {r.dice:.3f}] {r.image_id} {r.mode} {r.bone}"); warn += 1
print("warnings:", warn)

## 9. Figures
In the QC overlay, superior should be at the top and the contours should hug the femur. If the image is upside down or the contours sit on the other leg, you have an orientation problem.

In [ ]:
from IPython.display import Image
CASE = cases.image_id.iloc[0]
!python scripts/make_figures.py {DATA} --case {CASE} --preds preds --results results
for f in sorted(glob.glob("results/figures/*.png")):
    print(f); display(Image(f, width=700))

## 10. Save results
This zips the results (without the large NIfTI predictions), copies them to Drive, and downloads the zip.

In [ ]:
!zip -qr results.zip results cases.csv preds/runtimes.csv -x "results/stl/*"
if IN_COLAB and SAVE_TO_DRIVE:
    shutil.copy("results.zip", DRIVE_OUT)
    shutil.copytree("results", f"{DRIVE_OUT}/results", dirs_exist_ok=True)
    print("copied to", DRIVE_OUT)
if IN_COLAB:
    from google.colab import files; files.download("results.zip")
print(open("results/summary.md").read())

## Next steps
1. Paste the tables from `results/summary.md` into the README. Write 2–3 sentences on whether the hypothesis held, using your actual numbers.
2. List the limitations honestly: N, which split, model-assisted GT, the sphere-fit assumption, and hand-chosen thresholds.
3. Push the code + `results/` (not the data) to GitHub: see the project guide, section "Push to GitHub".
4. Only then add the CV entry, using **your real numbers**.